# 子图匹配训练流程演示

本笔记本演示训练子图匹配编码器的最小流程：解析参数、初始化运行时、构建模型、取一批数据并完成一次参数更新。为了便于在不同环境下直接运行，这里默认使用 `syn` 数据集并强制在 CPU 上执行。

In [1]:
import sys
from pathlib import Path

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repository root")
repo_root = str(repo_root)
sys.path.insert(0, repo_root)

import argparse
import torch

from src.core.cli import setup_runtime
from src.subgraph_matching.config import parse_encoder
from src.subgraph_matching import train as train_mod

In [2]:
parser = argparse.ArgumentParser(description="训练流程演示")
parse_encoder(parser)
args = parser.parse_args("")
args.use_gpu = False
args.dataset = "syn"
args.node_anchored = True
args.batch_size = 16
args.n_batches = 1
args.eval_interval = 1
args.val_size = 32
args.n_workers = 1
args.seed = 0
args.model_path = str(Path(repo_root) / "results/notebook-train-model.pt")
Path(args.model_path).parent.mkdir(parents=True, exist_ok=True)
device = setup_runtime(args)
print({"device": str(device), "dataset": args.dataset, "model_path": args.model_path})

{'device': 'cpu', 'dataset': 'syn', 'model_path': 'd:\\py\\Gnntest\\neural\\SPMiner_learn\\results\\notebook-train-model.pt'}


In [3]:
data_source = train_mod.make_data_source(args)
model = train_mod.build_model(args)
loaders = data_source.gen_data_loaders(args.eval_interval * args.batch_size, args.batch_size, train=True)
batch_target, batch_neg_target, batch_neg_query = next(iter(zip(*loaders)))
pos_a, pos_b, neg_a, neg_b = data_source.gen_batch(batch_target, batch_neg_target, batch_neg_query, True)
print({"pos_graphs": int(pos_a.num_graphs), "neg_graphs": int(neg_a.num_graphs)})

{'pos_graphs': 8, 'neg_graphs': 8}


d:\conda\envs\neural-subgraph-learning-GNN\lib\site-packages\deepsnap\graph.py:522: UserWarning: Node related key is required.
  warnings.warn("Node related key is required.")


In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
model.train()
optimizer.zero_grad()

emb_as = model.emb_model(pos_a)
emb_bs = model.emb_model(pos_b)
neg_as = model.emb_model(neg_a)
neg_bs = model.emb_model(neg_b)
emb_as = torch.cat([emb_as, neg_as], dim=0)
emb_bs = torch.cat([emb_bs, neg_bs], dim=0)
labels = torch.tensor([1] * pos_a.num_graphs + [0] * neg_a.num_graphs).to(device)
pred = model(emb_as, emb_bs)
loss = model.criterion(pred, None, labels)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
torch.save(model.state_dict(), args.model_path)
print({"loss": float(loss.detach().cpu()), "saved_to": args.model_path})

{'loss': 0.9689211845397949, 'saved_to': 'd:\\py\\Gnntest\\neural\\SPMiner_learn\\results\\notebook-train-model.pt'}
